In [70]:
import pandas as pd
import os
import re

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [71]:
def _fix_mojibake_text(x):
    if not isinstance(x, str):
        return x

    s = x
    if not any(ch in s for ch in ("Ã", "Â", "¤", "¦", "¥", "œ", "�")):
        return s

    candidates = [s]
    for enc in ("latin1", "cp1252"):
        try:
            candidates.append(s.encode(enc).decode("utf-8"))
        except Exception:
            pass

    def score(t):
        bad = sum(t.count(ch) for ch in ("Ã", "Â", "�"))
        return (bad, -sum(ord(ch) > 127 for ch in t))

    best = min(candidates, key=score)
    return best

p = 'AU/bio_data.csv'
df = pd.read_csv(p)
df_edited = df.copy()

for col in df_edited.select_dtypes(include=["object"]).columns:
    df_edited[col] = df_edited[col].map(_fix_mojibake_text)

out_p = os.path.join(os.path.dirname(p), "bio_data_edited.csv")
df_edited.to_csv(out_p, index=False, encoding="utf-8-sig")

print(f"Saved: {out_p}")

Saved: AU\bio_data_edited.csv


C:\Users\hamod4691\AppData\Local\Temp\ipykernel_17784\3691843218.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_edited.select_dtypes(include=["object"]).columns:


In [ ]:
p = 'AU/bio_data_edited.csv'
df = pd.read_csv(p)

qualifier_pattern = re.compile(
    r'\([^)]*\)|'                       # parentheses (keep them)
    r'(?i:\bPamiut|NKF\b)|'             # specific place-name qualifier
    r'(?i:\bsp\.?\d+\b)|'             # sp2 or sp.2
    r'(?i:(?<=\s)\.?juv\.?)|(?i:(?<=\s)juv\.?)(?=\W|$)|'  # juv preceded by space, dot optional
    r'(?i:gen\. et spp\.?)|'            # gen. et spp.
    r'(?i:\b(?:spp|sp|indet|lign|stage|conf|var)\.?)(?=\W|$)|'  # spp/sp/indet/lign/conf/var
    r'\bK\d(?:-\d){2}\b|'             # K-codes like K3-2-3
    r'\b[A-Z]{1,}[0-9]+(?:-[0-9]+)?\b|' # codes like NKF22-4, DW31-1, T-1 (letters+digits)
    r'\b[A-Z](?:-\d+|\d+)?\b|'        # lone capital letters maybe with -number (e.g. T-1)
    r'\b\d+\b|'                       # standalone numbers
    r'\?+'                              # question marks
)


def _extract_qualifiers(text):
    matches = list(qualifier_pattern.finditer(text))
    qualifiers = [match.group(0).strip() for match in matches]

    parts = []
    last_end = 0
    for match in matches:
        parts.append(text[last_end:match.start()])
        last_end = match.end()
    parts.append(text[last_end:])

    cleaned = re.sub(r'\s+', ' ', ''.join(parts)).strip(' ,;.-')
    if cleaned == '':
        cleaned = pd.NA

    return cleaned, qualifiers


def split_qualifier(s):
    if pd.isna(s):
        return pd.Series([pd.NA, pd.NA])

    orig = str(s)
    if '/' in orig:
        left, right = orig.split('/', 1)
        cleaned, qualifiers = _extract_qualifiers(left)
        right_cleaned, right_qualifiers = _extract_qualifiers(right)
        qualifiers.extend(right_qualifiers)
        if pd.notna(right_cleaned):
            qualifiers.append(str(right_cleaned).strip())
    else:
        cleaned, qualifiers = _extract_qualifiers(orig)

    qual = '; '.join(q for q in qualifiers if q and q.strip()) if qualifiers else pd.NA
    return pd.Series([cleaned, qual])


df[['scientificName', 'identificationQualifier']] = df['Taxon edited'].apply(split_qualifier)
# write CSV (may raise if file is locked)
df.to_csv(p, index=False)
# also write Excel
try:
    df.to_excel(os.path.splitext(p)[0] + '.xlsx', index=False)
except Exception as e:
    print('Excel write failed:', e)

In [ ]:
p = 'AU/bio_data_edited.csv'
df = pd.read_csv(p)

def _save_outputs():
    df.to_csv(p, index=False, encoding='utf-8-sig')
    df.to_excel(os.path.splitext(p)[0] + '.xlsx', index=False)

# Remove rows where scientificName is "ovae" or "ukendt" (case/space insensitive)
to_remove = {'ovae', 'ukendt', "megalopedia", "Zoea larvae", "Actinaridae"}
mask_remove = (
    df['scientificName']
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(to_remove)
)

df = df.loc[~mask_remove].copy()
_save_outputs()
print(f"Removed {mask_remove.sum()} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

rename_pairs = [
    ('Ampharete finmarchia', 'Ampharete finmarchica'),
    ('Ampelisca eschrichti', 'Ampelisca eschrichtii'),
    ('Ampharete goesi', 'Ampharete goësi'),
    ('Amphitite', 'Amphitrite'),
    ('Bivalvia ånderør', 'Bivalvia'),
    ('Byblis gaimardi', 'Byblis gaimardii'),
    ('Bylgides groenlandica', 'Bylgides groenlandicus'),
    ('Cingula moerchi', 'Alvania moerchii'),
    ('Cingula moercki', 'Alvania moerchii'),
    ('Cossura longicirrata', 'Cossura longocirrata'),
    ('Diastylis scorpipides', 'Diastylis scorpioides'),
    ('Microphthalmus sczelkowi', 'Microphthalmus sczelkowii'),
    ('Monoculodes kroyeri', 'Rostroculodes kroyeri'),
    ('Monoculodes krøyeri', 'Rostroculodes kroyeri'),
    ('Mya ånderør', 'Mya sp.'),
    ('Nainereis quadricuspida', 'Naineris quadricuspida'),
    ('Nebalia biceps', 'Nebalia bipes'),
    ('Nephtys longosetosus', 'Nephtys longosetosa'),
    ('Nicomache trispina', 'Nicomache trispinata'),
    ('Nudibranch', 'Nudibranchia'),
    ('Octocoral', 'Octocorallia'),
    ('Oediceros sagnisatus', 'Oediceros saginatus'),
    ('Oediceridae', 'Oedicerotidae'),
    ('Oenopota turricola', 'Propebela turricula'),
    ('Oenopota turricula', 'Propebela turricula'),
    ('Ophiocanta bidentata', 'Ophiacantha bidentata'),
    ('Ophiura sarsi', 'Ophiura sarsii'),
    ('Opisa eschrichti', 'Opisa eschrichtii'),
    ('Pholoë', 'Pholoe'),
    ('Pholoë', 'Pholoe'),
    ('Phoxocephalus holboelli', 'Phoxocephalus holbolli'),
    ('Phoxocephalus holbølli', 'Phoxocephalus holbolli'),
    ('Sphaerodoropsis minuta', 'Sphaerodoridium minutum'),
    ('Tachyrhyncus reticulatus', 'Tachyrhynchus reticulatus'),
    ('Thyasira gouldi', 'Thyasira gouldii'),
    ('Travisia forbesi', 'Travisia forbesii'),
    ('Urothoe elegans', 'Urthoe elegans'),
    ('Yoldia hyperboria', 'Yoldia hyperborea'),
]

for old_name, new_name in rename_pairs:
    mask = df['scientificName'].astype(str).str.strip().eq(old_name)
    count = mask.sum()
    df.loc[mask, 'scientificName'] = new_name
    _save_outputs()
    print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Removed 4 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 10 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 32 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 15 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 4 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 16 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx
Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.

In [74]:
unique_names = (
    df["scientificName"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_names_df = pd.DataFrame({"scientificName": unique_names})
unique_names_df.to_csv("AU/unique_names.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(unique_names_df)} unique names to unique_names.csv")
unique_names_df.head()

Saved 1144 unique names to unique_names.csv


,scientificName
0,Abyssoninoe
1,Abyssoninoe abyssorum
2,Acanthonotozoma magnum
3,Acanthonotozoma rusanovae
4,Acanthonotozoma serratum


In [75]:
stations = pd.read_csv('AU/stations_1.csv')
print(stations.columns.tolist())

station_coords = stations.loc[:, ['Cruise ID', 'Station ID', 'Depth', 'lat_dd', 'W long (dd.ddd).1']].copy()
station_coords = station_coords.rename(columns={
    'lat_dd': 'latitude_wgs84',
    'W long (dd.ddd).1': 'longitude_wgs84',
})
station_coords = station_coords.dropna(subset=['latitude_wgs84', 'longitude_wgs84'])

station_coords.to_csv('AU/stations_1_wgs84.csv', index=False, encoding='utf-8-sig')

print('Saved AU/stations_1_wgs84.csv')
station_coords.head()

['Cruise ID', 'Year', 'Month ', 'Day', 'Date', 'Station ID', 'Depth', 'N lat (dd°mm.mmm)', 'W long (dd°mm.mmm)', 'N lat (dd)', 'N lat (mm.mmm)', 'lat_dd', 'W long (dd)', 'W long (mm.mmm)', 'W long (dd.ddd)', 'W long (dd.ddd).1', 'sampling equip']
Saved AU/stations_1_wgs84.csv


,Cruise ID,Station ID,Depth,latitude_wgs84,longitude_wgs84
0,KANUMAS,1.1,21.0,76.920617,-70.907383
1,KANUMAS,1.2,65.0,76.915600,-71.061500
2,KANUMAS,3.1,22.0,76.263517,-68.989933
3,KANUMAS,3.2,60.0,76.241733,-69.099583
4,KANUMAS,4.1,19.0,76.156667,-68.483350
